# Lesson 7.4 — Unit 7 Recap

Adaptation = **pre-validation + action-selection** inside the **twin-in-the-loop** cycle. No ML, RL, adaptive control, or optimization.

In [1]:
import numpy as np
from modules.module09.integration import GreenhouseWorld, harvest_row
from modules.module10.twin import (GroundTruth, DigitalTwin, snapshot,
    prevalidate, select_action, twin_in_the_loop)
mk = lambda: GreenhouseWorld.demo_row(n=5, seed=1)
F2 = mk().fruit[2]
OBST = {'F2': {'obstacle': ((float(F2.x), float(F2.y)), 0.05)}}
def effort(rep): return sum(p['n_attempts'] for p in rep['picks'])
checks = []
twin = DigitalTwin(mk()); state = snapshot(mk()); twin.sync(state)
# (1) pre-validate: accept clean, reject blocked
checks.append(prevalidate(twin, None)['accept'] is True)
checks.append(prevalidate(twin, OBST)['accept'] is False)


In [2]:
# (2) choose the better action
sel = select_action(twin, {'clear_path': None, 'ignore_obstacle': OBST})
checks.append(sel['chosen'] == 'clear_path')


In [3]:
# (3) close the loop: in-sync (quiet) and drifted (alert + resync)
t1 = twin_in_the_loop(twin, state, {'clear_path': None, 'ignore_obstacle': OBST})
real = GroundTruth(mk()); real.world.q = np.asarray(real.world.q, float) + np.array([0.08, -0.04])
t2 = twin_in_the_loop(twin, snapshot(real.world), {'clear_path': None, 'ignore_obstacle': OBST})
checks.append(t1['resynced'] is False and t2['resynced'] is True)
print('pre-validate + choose + loop all consistent')


pre-validate + choose + loop all consistent


In [4]:
assert all(checks), checks
print('All checks passed.')


All checks passed.
